# 🛰️ Monitoramento de Queimadas em UCs do Pará (NGEO / IDEFLOR-Bio)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NGEO-IDEFLOR-Bio/cartografia-queimadas-ucs-pa/blob/main/notebooks/colab_runner.ipynb)

Painel automatizado para detecção de focos de calor (**BDQueimadas / INPE**) e análise de cicatrizes de fogo com **Sentinel-2 Harmonized (Google Earth Engine)** nas 29 Unidades de Conservação Estaduais do Pará.

Gera automaticamente 3 pranchas no padrão oficial **A4 Paisagem (297 x 210 mm)**:
1. **Falsa Cor (SWIR-NIR-Red)**: Realce de cicatrizes recentes e frentes de queima ativa.
2. **Cor Natural (RGB)**: Mosaico orbital sem nuvens.
3. **NBR (Normalized Burn Ratio)**: Índice espectral de severidade do fogo.

In [ ]:
#@title 1. Instalação de Dependências e Setup do Ambiente { display-mode: "form" }
#@markdown *Clique no botão 'Play' para preparar o ambiente e baixar o repositório.*

!apt-get update -qq && apt-get install -y -qq libreoffice-writer > /dev/null 2>&1
!pip install -q earthengine-api geopandas rasterio matplotlib pillow python-docx openpyxl

import os
import sys
from pathlib import Path

REPO_NAME = "cartografia-queimadas-ucs-pa"
if not Path(REPO_NAME).exists() and not Path("src").exists():
    print(f"Clonando o repositório oficial {REPO_NAME}...")
    !git clone https://github.com/NGEO-IDEFLOR-Bio/cartografia-queimadas-ucs-pa.git

if Path(REPO_NAME).exists():
    os.chdir(REPO_NAME)

sys.path.append(os.getcwd())
print(f"[OK] Ambiente preparado e repositório configurado: {os.getcwd()}")


In [ ]:
#@title 2. Autenticação no Google Earth Engine (GEE) { display-mode: "form" }
GCP_PROJECT = "ee-samuelsantosambientalcourse" #@param {type:"string"}
#@markdown > ⚠️ **Atenção colegas:** Altere o campo acima para o ID do seu próprio projeto Google Cloud habilitado no Earth Engine.

import ee

try:
    ee.Initialize(project=GCP_PROJECT)
    print(f"[OK] Google Earth Engine inicializado com sucesso no projeto: {GCP_PROJECT}")
except Exception as e:
    print(f"! Solicitando autorização de acesso ao GEE...")
    ee.Authenticate()
    ee.Initialize(project=GCP_PROJECT)
    print(f"[OK] Google Earth Engine autenticado e inicializado com sucesso no projeto: {GCP_PROJECT}!")

In [ ]:
#@title 3. Parâmetros de Análise Cartográfica { display-mode: "form" }
NOME_UC = "Área de Proteção Ambiental do Marajó" #@param ["Área de Proteção Ambiental do Marajó", "Área de Proteção Ambiental da Ilha de Combu", "Área de Proteção Ambiental do Lago de Tucuruí", "Área de Proteção Ambiental Triunfo do Xingu", "Área de Proteção Ambiental de Algodoal-Maiandeua", "Área de Proteção Ambiental de São Geraldo do Araguaia – APA Araguaia", "Área de Proteção Ambiental da Região Metropolitana de Belém", "Área de Proteção Ambiental Paytuna", "Parque Estadual Monte Alegre – PEMA", "Parque Estadual Charapucu", "Parque Estadual da Serra dos Martírios/Andorinhas", "Parque Estadual do Utinga “Camillo Vianna”", "Parque Estadual Ambiental das Árvores Gigantes da Amazônia (PAGAM)", "Floresta Estadual do Paru", "Floresta Estadual do Trombetas", "Floresta Estadual do Iriri", "Flota Estadual de Faro", "Estação Ecológica do Grão-Pará (ESEC)", "Estação Ecológica Mamuru (ESEC)", "Reserva Biológica do Maicuru (REBIO)", "Refúgio de Vida Silvestre Metrópole da Amazônia (REVIS)", "Refúgio de Vida Silvestre Padre Sérgio Tonetto", "Refúgio de Vida Silvestre Rios São Benedito e Azul  (REVIS)", "Refúgio de Vida Silvestre Tabuleiro do Embaubal (REVIS)", "Reserva de Desenvolvimento Sustentável Alcobaça", "Reserva de Desenvolvimento Sustentável Campo das Mangabas", "Reserva de Desenvolvimento Sustentável Pucuruí-Ararão", "Reserva de Desenvolvimento Sustentável Vitória de Souzel", "Monumento Natural Atalaia (MONA)"] {allow-input: true}
DIAS_MONITORAMENTO = 30 #@param {type:"slider", min:7, max:30, step:1}
#@markdown *O monitoramento analisa automaticamente a janela recente (ex: últimos 30 dias até hoje) direto do INPE/BDQueimadas sem necessidade de baixar arquivos.*

from datetime import date, timedelta
hoje = date.today()
data_inicio_calc = (hoje - timedelta(days=DIAS_MONITORAMENTO)).strftime("%d/%m/%Y")
data_fim_calc = hoje.strftime("%d/%m/%Y")

print(f"Configuração ativa:")
print(f" • Unidade de Conservação: {NOME_UC}")
print(f" • Janela de Monitoramento: Últimos {DIAS_MONITORAMENTO} dias ({data_inicio_calc} até {data_fim_calc})")
print(f" • Fonte dos Focos: BDQueimadas/INPE (Download 100% Automático)")

In [ ]:
#@title 4. Executar Pipeline e Gerar as Pranchas Oficiais { display-mode: "form" }
#@markdown *Clique no 'Play' para iniciar o processamento integrado (Sentinel-2 + BDQueimadas + Layout A4 + Relatório Word + Excel + KML).*

from src.exporter import processar_e_exportar_todos_os_mapas

resultado = processar_e_exportar_todos_os_mapas(
    nome_uc=NOME_UC,
    dias_analise=DIAS_MONITORAMENTO,
    focos_source="BDQUEIMADAS"
)


In [ ]:
#@title 5. Visualizar Pranchas Cartográficas e Arquivos Gerados { display-mode: "form" }

if 'resultado' not in globals():
    raise RuntimeError("⚠️ Atenção: A etapa 4 não foi executada ainda. Por favor, clique no Play da Célula '4. Executar Pipeline' antes de visualizar os mapas.")

from pathlib import Path
from IPython.display import Image, display

pranchas = resultado.get("pranchas", {})
uc_nome = resultado.get("uc", "Unidade de Conservação")

print("\n" + "="*70)
print(f"🗺️ PRANCHAS CARTOGRÁFICAS GERADAS – {uc_nome.upper()}")
print("="*70)

for chave, info in pranchas.items():
    titulo_mapa = info["titulo"].upper()
    png_path = info.get("png", "")
    pdf_path = info.get("pdf", "")
    print(f"\n--- {titulo_mapa} ---")
    print(f"  • Imagem PNG (300 DPI): {png_path}")
    print(f"  • Prancha PDF Vetorial: {pdf_path}")
    if png_path and Path(png_path).exists():
        display(Image(filename=png_path, width=950))

print("\n" + "="*70)
print("📦 ARQUIVOS TÉCNICOS E ESPACIAIS GERADOS (PASTA outputs/)")
print("="*70)

rel_docx = resultado.get("relatorio_docx")
rel_pdf = resultado.get("relatorio_pdf")
planilha_xlsx = resultado.get("planilha_excel")
kml_path = resultado.get("pacote_kml")

if rel_docx:
    print(f"  • Relatório Técnico Word (.docx): {rel_docx}")
if rel_pdf:
    print(f"  • Relatório Técnico PDF (.pdf):   {rel_pdf}")
if planilha_xlsx:
    print(f"  • Planilha Eletrônica de Focos (.xlsx): {planilha_xlsx}")
if kml_path:
    print(f"  • Pacote Google Earth / GPS (.kml):    {kml_path}")

print("\nNota: Todos os arquivos acima estão salvos na pasta 'outputs/' do repositório.")
print("      No Google Colab, use o painel lateral de arquivos para baixá-los diretamente.")


In [ ]:
#@title 6. (Opcional) Baixar Pacote Completo para seu Computador { display-mode: "form" }
#@markdown *Execute esta célula para iniciar o download direto de todos os arquivos gerados.*

if 'resultado' not in globals():
    raise RuntimeError("⚠️ Execute a Célula 4 antes de tentar baixar os arquivos.")

try:
    from google.colab import files
    from pathlib import Path
    
    arquivos = [
        resultado.get("relatorio_docx"),
        resultado.get("relatorio_pdf"),
        resultado.get("planilha_excel"),
        resultado.get("pacote_kml"),
    ]
    for p_info in resultado.get("pranchas", {}).values():
        arquivos.append(p_info.get("pdf"))
        arquivos.append(p_info.get("png"))
        
    for f in arquivos:
        if f and Path(f).exists():
            print(f"Iniciando download: {Path(f).name}...")
            files.download(f)
except Exception as e:
    print(f"Ambiente local ou download interativo ignorado: {e}")
